In [1]:
#STEP 0: Paths

from pathlib import Path
import re

from lxml import etree
from svgutils.compose import Figure, SVG, Text

def find_project_root(start=None):
    """
    Find the repository root by locating both environment.yml and .git.
    """
    path = Path(start or Path.cwd()).resolve()

    for parent in [path, *path.parents]:
        if ((parent / "environment.yml").exists() and (parent / ".git").exists()):
            return parent

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure this notebook is running inside the repository.")

#Project root and analysis environment
PROJECT_ROOT = find_project_root()
env = "ENV1"

#Main directories

ADATA_DIR = PROJECT_ROOT / "Adata_Saves"
DATAFRAME_DIR = PROJECT_ROOT / "Dataframe_Saves"
INPUT_DIR = PROJECT_ROOT / "Input_Dataframes"

SCVI_MODEL_DIR = PROJECT_ROOT / "scvi_model"
SCVI_TUMOUR_MODEL_DIR = PROJECT_ROOT / "scvi_tumour_model"

FIGURE_DIR = PROJECT_ROOT / "Figures"
FIGDATA_DIR = PROJECT_ROOT / "Figure_Data"


#Environment-specific directories
ENV_DIR = ADATA_DIR / env
HEAD_DIR = ENV_DIR / "head"
ENV_FIGURE_DIR = FIGURE_DIR / env

# Create required directories
DIRECTORIES = [ADATA_DIR, DATAFRAME_DIR, INPUT_DIR, SCVI_MODEL_DIR, SCVI_TUMOUR_MODEL_DIR, FIGURE_DIR, FIGDATA_DIR, ENV_DIR, HEAD_DIR, ENV_FIGURE_DIR]
for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

#Print paths for reproducibility/debugging
print(f"Project root:        {PROJECT_ROOT}")
print(f"Environment:         {env}")
print(f"AnnData directory:   {ADATA_DIR}")
print(f"Environment data:    {ENV_DIR}")
print(f"Head data:           {HEAD_DIR}")
print(f"Dataframe directory: {DATAFRAME_DIR}")
print(f"Input directory:     {INPUT_DIR}")
print(f"Figure directory:    {FIGURE_DIR}")
print(f"Environment figures: {ENV_FIGURE_DIR}")
print(f"Figure data:         {FIGDATA_DIR}")

#STEP 1: PRISM SVGs
#PRISM SVGs live in Input_Dataframes/PRISM
PRISM_DIR = INPUT_DIR / "PRISM"

#Standardized SVGs will be stored separately
PRISM_STD_DIR = PRISM_DIR / "standardized"
PRISM_DIR.mkdir(parents=True, exist_ok=True)
PRISM_STD_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPRISM directory:     {PRISM_DIR}")
print(f"Standardized SVGs:   {PRISM_STD_DIR}")

#Check that the expected SVGs exist
PLOT_NAMES = [f"Plot{i}" for i in range(1, 7)]
missing_plots = [
    name
    for name in PLOT_NAMES
    if not (PRISM_DIR / f"{name}.svg").exists()
]

if missing_plots:
    raise FileNotFoundError(
        "The following expected PRISM SVGs are missing:\n"
        + "\n".join(f"  - {name}.svg" for name in missing_plots)
    )

print("\nAvailable PRISM SVGs:")
for svg_file in sorted(PRISM_DIR.glob("*.svg")):
    print(f"  {svg_file.name}")


Project root:        /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny
Environment:         ENV1
AnnData directory:   /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Adata_Saves
Environment data:    /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Adata_Saves/ENV1
Head data:           /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Adata_Saves/ENV1/head
Dataframe directory: /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Dataframe_Saves
Input directory:     /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Input_Dataframes
Figure directory:    /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Figures
Environment figures: /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Figures/ENV1
Figure data:         /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Figure_Data

PRISM directory:     /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Input_Dataframes/PRISM
Standardized SVGs:   /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Input_Dataframes/PRISM/standardized

Available PRISM SVGs:
  ._Figure_prism_combined.svg
  ._Plot1.svg
  ._Plot2.svg
  ._Plot3.svg
  ._Plot4

In [2]:

# =============================================================================
# STEP 2: SVG dimensions
# =============================================================================

SVG_NS = "http://www.w3.org/2000/svg"
NSMAP = {"svg": SVG_NS}


def get_native_size(svg_path):
    """
    Return the native width and height of an SVG.

    Preference:
        1. viewBox dimensions
        2. width/height attributes

    Returns
    -------
    tuple[float, float]
        Width and height in the SVG's native coordinate system.
    """
    svg_path = Path(svg_path)

    tree = etree.parse(str(svg_path))
    root = tree.getroot()

    # Prefer viewBox because this defines the SVG coordinate system
    viewbox = root.get("viewBox")

    if viewbox:
        values = viewbox.replace(",", " ").split()

        if len(values) == 4:
            _, _, width, height = map(float, values)
            return width, height

    # Fallback to width/height attributes
    def parse_dimension(value, attribute_name):
        if value is None:
            raise ValueError(
                f"{svg_path} has no viewBox and no {attribute_name} attribute."
            )

        match = re.match(r"^\s*([0-9.+-]+)", value)

        if match is None:
            raise ValueError(
                f"Could not parse {attribute_name}='{value}' "
                f"in {svg_path}"
            )

        return float(match.group(1))

    width = parse_dimension(root.get("width"), "width")
    height = parse_dimension(root.get("height"), "height")

    return width, height


# -----------------------------------------------------------------------------
# Read native dimensions
# -----------------------------------------------------------------------------

native = {
    name: get_native_size(PRISM_DIR / f"{name}.svg")
    for name in PLOT_NAMES
}

print("\nNative SVG dimensions:")

for name, (width, height) in native.items():
    print(f"  {name}: {width:.1f} × {height:.1f}")
LABEL_SIZE = 8
PANEL_TITLE_SIZE = 8
ROW_TITLE_SIZE = 8


Native SVG dimensions:
  Plot1: 266.0 × 174.0
  Plot2: 273.0 × 177.0
  Plot3: 273.0 × 177.0
  Plot4: 273.0 × 176.0
  Plot5: 276.0 × 182.0
  Plot6: 356.0 × 496.0


In [3]:
# =============================================================================
# STEP 3: Figure layout settings
# =============================================================================

# -----------------------------------------------------------------------------
# Main layout controls
# -----------------------------------------------------------------------------

# Plots 1–5 are placed inside identical square panels.
#
# Change DIAGRAM_SIZE to make ALL five diagrams larger or smaller.
#
# Examples:
#     180 = smaller
#     200 = current
#     220 = larger
#     240 = very large
#
DIAGRAM_SIZE = 200

# Horizontal spacing between Plots 1–5
HORIZONTAL_GAP = 90

# Vertical spacing between rows
VERTICAL_GAP = 15

# Plot 6 uses the same panel width as Plots 1–5
PLOT6_WIDTH = DIAGRAM_SIZE

SHARED_X_LABEL_HEIGHT = 25

# -----------------------------------------------------------------------------
# Typography
# -----------------------------------------------------------------------------

# One font size throughout the entire figure
FONT_SIZE = 10

# Vertical spacing between lines of a two-line title
LINE_HEIGHT = 5

# Space reserved for the row title
ROW_TITLE_HEIGHT = 25

# Space reserved for two-line panel titles
PANEL_TITLE_HEIGHT = (2 * LINE_HEIGHT)


# -----------------------------------------------------------------------------
# Overall figure width
# -----------------------------------------------------------------------------

GRID_WIDTH = (3 * DIAGRAM_SIZE + 2 * HORIZONTAL_GAP)


print("\nFigure settings")
print("-" * 60)
print(f"Diagram size:       {DIAGRAM_SIZE} × {DIAGRAM_SIZE}")
print(f"Horizontal gap:     {HORIZONTAL_GAP}")
print(f"Vertical gap:       {VERTICAL_GAP}")
print(f"Font size:          {FONT_SIZE} pt")
print(f"Grid width:         {GRID_WIDTH}")
print(f"Plot 6 width:       {PLOT6_WIDTH}")

# =============================================================================
# STEP 4: Detect the actual Prism plotting area
# =============================================================================
def get_plot_area(svg_path):
    """
    Detect the rectangular Prism plotting area from an SVG clipPath.

    Prism commonly stores the graph region as a rectangular <path>,
    for example:

        M x1 y1
        L x2 y1
        L x2 y2
        L x1 y2
        Z

    Returns
    -------
    dict
        x_min, x_max, y_min, y_max, width, height
    """

    svg_path = Path(svg_path)

    tree = etree.parse(str(svg_path))
    root = tree.getroot()

    candidates = []

    for clip_path in root.findall(".//svg:clipPath", NSMAP):

        for element in clip_path.iter():

            if etree.QName(element).localname != "path":
                continue

            d = element.get("d")

            if not d:
                continue

            # -------------------------------------------------------------
            # Extract coordinate pairs following M/L commands
            # -------------------------------------------------------------

            pattern = re.compile(
                r"[ML]\s*"
                r"([-+]?(?:\d*\.\d+|\d+\.?))"
                r"[\s,]+"
                r"([-+]?(?:\d*\.\d+|\d+\.?))",
                flags=re.IGNORECASE,
            )

            points = [
                (float(x), float(y))
                for x, y in pattern.findall(d)
            ]

            if len(points) < 4:
                continue

            x_values = [x for x, y in points]
            y_values = [y for x, y in points]

            x_min = min(x_values)
            x_max = max(x_values)

            y_min = min(y_values)
            y_max = max(y_values)

            width = x_max - x_min
            height = y_max - y_min

            # Ignore tiny clip regions
            if width < 20 or height < 20:
                continue

            candidates.append(
                {
                    "x_min": x_min,
                    "x_max": x_max,
                    "y_min": y_min,
                    "y_max": y_max,
                    "width": width,
                    "height": height,
                }
            )

    if not candidates:
        raise ValueError(
            f"Could not identify a rectangular plotting area in:\n"
            f"{svg_path}\n\n"
            f"Available clipPaths: "
            f"{len(root.findall('.//svg:clipPath', NSMAP))}"
        )

    # Largest rectangular candidate = main plotting region
    plot_area = max(
        candidates,
        key=lambda x: x["width"] * x["height"],
    )

    return plot_area
# -----------------------------------------------------------------------------
# Detect plot areas for Plots 1–5
# -----------------------------------------------------------------------------

plot_areas = {
    name: get_plot_area(
        PRISM_DIR / f"{name}.svg"
    )
    for name in PLOT_NAMES[:5]
}

print("\nDetected Prism plotting areas:")
print("-" * 70)

for name, area in plot_areas.items():

    print(
        f"{name}: "
        f"x = {area['width']:.2f}, "
        f"y = {area['height']:.2f}, "
        f"aspect = {area['width'] / area['height']:.3f}"
    )
# =============================================================================
# Plot 1 = reference graph size
# =============================================================================

REFERENCE_PLOT = "Plot1"

REFERENCE_WIDTH = plot_areas[
    REFERENCE_PLOT
]["width"]

REFERENCE_HEIGHT = plot_areas[
    REFERENCE_PLOT
]["height"]


print("\nReference plot:")
print(f"  {REFERENCE_PLOT}")
print(f"  Width:  {REFERENCE_WIDTH:.2f}")
print(f"  Height: {REFERENCE_HEIGHT:.2f}")

# =============================================================================
# Calculate scaling factors
# =============================================================================

scale_factors = {}

for name in PLOT_NAMES[:5]:          # ← only Plot1–Plot5, Plot6 excluded on purpose
    current_width = plot_areas[name]["width"]
    scale_factors[name] = REFERENCE_WIDTH / current_width

plot_areas["Plot6"] = get_plot_area(PRISM_DIR / "Plot6.svg")
scale_factors["Plot6"] = PLOT6_WIDTH / plot_areas["Plot6"]["width"]
# =============================================================================
# Check resulting axis dimensions
# =============================================================================

print("\nAxis dimensions after scaling:")
print("-" * 70)

for name in PLOT_NAMES[:5]:

    scale = scale_factors[name]

    scaled_width = (
            plot_areas[name]["width"]
            * scale
    )

    scaled_height = (
            plot_areas[name]["height"]
            * scale
    )

    print(
        f"{name}: "
        f"x = {scaled_width:.2f}, "
        f"y = {scaled_height:.2f}"
    )

# =============================================================================
# STEP 5: Standardize Prism fonts
# =============================================================================

def standardize_fonts_scaled(
        svg_path,
        out_path,
        target_fontsize,
        scale_factor,
        family="Arial",
):
    effective_size = target_fontsize / scale_factor

    tree = etree.parse(str(svg_path))
    root = tree.getroot()

    text_elements = (
            root.findall(".//svg:text", NSMAP)
            + root.findall(".//svg:tspan", NSMAP)
    )

    for element in text_elements:
        style = element.get("style")
        if style:
            properties = {}
            for item in style.split(";"):
                if ":" not in item:
                    continue
                key, value = item.split(":", 1)
                properties[key.strip()] = value.strip()

            properties["font-size"] = f"{effective_size:.2f}"   # ← no "pt" suffix
            properties["font-family"] = family

            element.set("style", ";".join(f"{k}:{v}" for k, v in properties.items()))
        else:
            element.set("font-size", f"{effective_size:.2f}")   # ← no "pt" suffix
            element.set("font-family", family)

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    tree.write(str(out_path), encoding="UTF-8", xml_declaration=True)


# -----------------------------------------------------------------------------
# Standardize each Prism SVG exactly once
# -----------------------------------------------------------------------------
for name, scale in scale_factors.items():   # ← only iterates Plot1–Plot5
    standardize_fonts_scaled(
        svg_path=PRISM_DIR / f"{name}.svg",
        out_path=PRISM_STD_DIR / f"{name}.svg",
        target_fontsize=FONT_SIZE,
        scale_factor=scale,
        family="Arial",
    )


print("\nStandardized SVGs written to:")
print(f"  {PRISM_STD_DIR}")


# =============================================================================
# STEP 6: Figure content
# =============================================================================
FIGURE_MARGIN = 55   # etwas über dem größten negativen Offset (52.2), als Sicherheitspuffer

TITLES = {
    "Plot1": "CD8 T cells for VH Epcam / VL Her2",
    "Plot2": "Jurkat NFAT for VH Epcam / VL Her2",
    "Plot3": "Jurkat NFAT with VH Epcam / VL Her2",
    "Plot4": "Jurkat NFAT with VH Epcam / VL Her2",
    "Plot5": "Jurkat NFAT with MDA-MB-468",
    "Plot6": "ROR1 / EGFR",
}


ROW_TITLES = {
    "a": "b",
    "b": "c",
    "c": "d",
}

# =============================================================================
# STEP 6.5: Custom legends
# =============================================================================

LEGEND_DIR = PRISM_DIR / "legends"
LEGEND_HEIGHT = 26

LEGEND_LABELS = {
    "Plot1": ("CHO Her2 / EpCAM", "CHO EGFR / EpCAM"),   # (top/black-dot label, bottom/white-dot label)
    "Plot2": ("CHO Her2 / EpCAM", "CHO EGFR / EpCAM"),
    "Plot3": ("MDA-MB-453", "MDA-MB-231"),
    "Plot4": ("T47D no blocking", "T47D + Trastuzumab"),
    "Plot5": ("antiEGFR / anti EpCAM", "antiHER2 / antiEpCAM"),
}

def make_legend_svg(out_path, top_label, bottom_label,
                    width=140, height=26, font_size=10, dot_radius=2.5,
                    line_half_length=8):
    cx = 10
    y_top = 7
    y_bottom = height - 7

    svg_content = f'''<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
  <line x1="{cx - line_half_length}" y1="{y_top}" x2="{cx + line_half_length}" y2="{y_top}" stroke="black" stroke-width="1"/>
  <circle cx="{cx}" cy="{y_top}" r="{dot_radius}" fill="black" stroke="black" stroke-width="1"/>

  <line x1="{cx - line_half_length}" y1="{y_bottom}" x2="{cx + line_half_length}" y2="{y_bottom}" stroke="black" stroke-width="1"/>
  <circle cx="{cx}" cy="{y_bottom}" r="{dot_radius}" fill="white" stroke="black" stroke-width="1"/>

  <text x="{cx + line_half_length + 6}" y="{y_top+3}" font-size="{font_size}" font-family="Arial">{top_label}</text>
  <text x="{cx + line_half_length + 6}" y="{y_bottom+3}" font-size="{font_size}" font-family="Arial">{bottom_label}</text>
</svg>'''

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        f.write(svg_content)


# generate once, before place_row() needs them
for name, (top, bottom) in LEGEND_LABELS.items():
    make_legend_svg(LEGEND_DIR / f"{name}_legend.svg", top, bottom, width=60)

print(f"Legends written to: {LEGEND_DIR}")
# =============================================================================
# STEP 7: Compose figure
# =============================================================================

svgs = []
texts = []


def add_multiline_title(
        text,
        x,
        y,
        weight="normal",
):
    """
    Add a multiline title using the global 7 pt font size.
    """

    for i, line in enumerate(text.split("\n")):

        texts.append(
            Text(
                line,
                x,
                y + i * LINE_HEIGHT,
                size=FONT_SIZE,
                weight=weight,
                font="Arial",
            )
        )


def add_row_title(
        text,
        x,
        y,
):
    """
    Add a row title using the global 7 pt font size.
    """

    texts.append(
        Text(
            text,
            x,
            y,
            size=FONT_SIZE,
            weight="bold",
            font="Arial",
        )
    )

def add_shared_x_label(text, row_center_x, y):
    texts.append(
        Text(
            text,
            row_center_x,
            y,
            size=FONT_SIZE,
            font="Arial",
        )
    )




Figure settings
------------------------------------------------------------
Diagram size:       200 × 200
Horizontal gap:     90
Vertical gap:       15
Font size:          10 pt
Grid width:         780
Plot 6 width:       200

Detected Prism plotting areas:
----------------------------------------------------------------------
Plot1: x = 217.05, y = 145.78, aspect = 1.489
Plot2: x = 216.34, y = 145.06, aspect = 1.491
Plot3: x = 216.34, y = 145.06, aspect = 1.491
Plot4: x = 216.34, y = 145.06, aspect = 1.491
Plot5: x = 216.53, y = 145.19, aspect = 1.491

Reference plot:
  Plot1
  Width:  217.05
  Height: 145.78

Axis dimensions after scaling:
----------------------------------------------------------------------
Plot1: x = 217.05, y = 145.78
Plot2: x = 217.05, y = 145.54
Plot3: x = 217.05, y = 145.54
Plot4: x = 217.05, y = 145.54
Plot5: x = 217.05, y = 145.54

Standardized SVGs written to:
  /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Input_Dataframes/PRISM/standardized
Legends written 

In [4]:
# =============================================================================
# STEP 7: Function to place a row of plots
# =============================================================================

def place_row(plot_names, row_letter, y_start):

    # Row title
    add_row_title(
        ROW_TITLES[row_letter],
        0 + FIGURE_MARGIN,
        y_start + FONT_SIZE + FIGURE_MARGIN,
    )

    row_content_y = y_start + ROW_TITLE_HEIGHT
    plot_y = row_content_y + PANEL_TITLE_HEIGHT

    for i, name in enumerate(plot_names):

        width, height = native[name]
        scale = scale_factors[name]

        display_width = width * scale
        display_height = height * scale

        cell_x = i * (DIAGRAM_SIZE + HORIZONTAL_GAP)

        # ---------------------------------------------------------------------
        # NO individual plot titles
        # ---------------------------------------------------------------------
        # add_multiline_title(...) has been removed

        # ---------------------------------------------------------------------
        # Center SVG within its panel
        # ---------------------------------------------------------------------
        x = (
                cell_x
                + (DIAGRAM_SIZE - display_width) / 2
                + FIGURE_MARGIN
        )

        y = (
                plot_y
                + (DIAGRAM_SIZE - display_height) / 2
                + FIGURE_MARGIN
        )

        # ---------------------------------------------------------------------
        # Plot 6 may not be in PRISM_STD_DIR
        # ---------------------------------------------------------------------
        if name == "Plot6":
            svg_path = PRISM_DIR / f"{name}.svg"
        else:
            svg_path = PRISM_STD_DIR / f"{name}.svg"

        svgs.append(
            SVG(svg_path)
            .scale(scale)
            .move(x, y)
        )

        # ---------------------------------------------------------------------
        # Custom legends ONLY for Plot 4 and Plot 5
        # ---------------------------------------------------------------------
        if name in ["Plot4", "Plot5"]:

            area = plot_areas[name]
            legend_gap = 4

            legend_x = (
                    x
                    + area["x_min"] * scale
                    + legend_gap
            )

            legend_y = (
                    y
                    + area["y_min"] * scale
                    - 30
            )

            svgs.append(
                SVG(
                    LEGEND_DIR / f"{name}_legend.svg"
                ).move(legend_x, legend_y)
            )

    return (
            ROW_TITLE_HEIGHT
            + PANEL_TITLE_HEIGHT
            + DIAGRAM_SIZE
            + LEGEND_HEIGHT
            + SHARED_X_LABEL_HEIGHT
    )


# =============================================================================
# STEP 8: Rows A and B
# =============================================================================

y = 0


# -----------------------------------------------------------------------------
# Row A
# -----------------------------------------------------------------------------

y += place_row(["Plot1", "Plot2", "Plot3"], "a", y)

# Shared X-axis label for Row A
add_shared_x_label(
    "Hemibody concentration [nm]",
    FIGURE_MARGIN + GRID_WIDTH / 2,
    y + 18,
)


# -----------------------------------------------------------------------------
# Row B
# -----------------------------------------------------------------------------

y += VERTICAL_GAP

y += place_row(
    ["Plot4", "Plot5", "Plot6"],
    "b",
    y,
)

# Shared X-axis label for Row B
add_shared_x_label(
    "Hemibody concentration [nm]",
    FIGURE_MARGIN + GRID_WIDTH / 2,
    y + 18,
)

In [5]:

# =============================================================================
# STEP 10: Save final figure
# =============================================================================

OUTPUT_SVG = (
        PRISM_DIR
        / "Figure_prism_combined.svg"
)

TOTAL_WIDTH = GRID_WIDTH + 2 * FIGURE_MARGIN
TOTAL_HEIGHT = y + FIGURE_MARGIN


Figure(
    str(TOTAL_WIDTH),
    str(TOTAL_HEIGHT),
    *svgs,
    *texts,
).save(
    str(OUTPUT_SVG)
)


# =============================================================================
# Final report
# =============================================================================

print("\n" + "=" * 70)
print("Combined figure created successfully")
print("=" * 70)

print(f"Output:            {OUTPUT_SVG}")
print(f"Figure width:      {TOTAL_WIDTH:.1f}")
print(f"Figure height:     {TOTAL_HEIGHT:.1f}")
print(
    f"Aspect ratio:      "
    f"{TOTAL_WIDTH / TOTAL_HEIGHT:.2f}"
)


Combined figure created successfully
Output:            /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny/Input_Dataframes/PRISM/Figure_prism_combined.svg
Figure width:      890.0
Figure height:     642.0
Aspect ratio:      1.39


In [6]:
for name in ["Plot1", "Plot2", "Plot3", "Plot4", "Plot5"]:
    w, h = native[name]
    scale = scale_factors[name]
    display_width = w * scale
    print(f"{name}: display_width={display_width:.1f} vs DIAGRAM_SIZE={DIAGRAM_SIZE} "
          f"→ x-offset would be {(DIAGRAM_SIZE - display_width)/2:.1f}")

Plot1: display_width=266.0 vs DIAGRAM_SIZE=200 → x-offset would be -33.0
Plot2: display_width=273.9 vs DIAGRAM_SIZE=200 → x-offset would be -36.9
Plot3: display_width=273.9 vs DIAGRAM_SIZE=200 → x-offset would be -36.9
Plot4: display_width=273.9 vs DIAGRAM_SIZE=200 → x-offset would be -36.9
Plot5: display_width=276.7 vs DIAGRAM_SIZE=200 → x-offset would be -38.3
